In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


GPU: Tesla T4 is available.
Number of GPUs available: 1


In [3]:
d_model = 4
key_dimension = 2

In [20]:
w_q = nn.Linear(d_model, key_dimension)
w_k = nn.Linear(d_model, key_dimension)
w_v = nn.Linear(d_model, key_dimension)
w_o = nn.Linear(key_dimension, d_model)

In [ ]:
dummy_input = torch.rand(2, 3, 4)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 4])


tensor([[[0.8164, 0.4527, 0.1434, 0.3966],
         [0.0861, 0.2002, 0.8317, 0.1501],
         [0.6365, 0.8037, 0.1662, 0.8496]],

        [[0.7670, 0.0584, 0.2540, 0.5282],
         [0.8294, 0.1127, 0.8117, 0.3630],
         [0.1884, 0.1756, 0.2164, 0.7477]]])

In [55]:
q = w_q(dummy_input)
k = w_k(dummy_input)
v = w_v(dummy_input)

In [ ]:
print(q.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 2])


tensor([[[-0.4443,  0.0985],
         [-0.0534,  0.6479],
         [-0.2622,  0.1610]],

        [[-0.5731,  0.0373],
         [-0.4541,  0.3184],
         [-0.2910,  0.1990]]], grad_fn=<ViewBackward0>)

In [57]:
print(k.shape)
k

torch.Size([2, 3, 2])


tensor([[[-0.1486,  0.8446],
         [ 0.3547,  0.9082],
         [-0.3911,  0.9802]],

        [[-0.0804,  0.7267],
         [ 0.1811,  0.9749],
         [-0.1462,  0.6828]]], grad_fn=<ViewBackward0>)

In [ ]:
k_t = k.transpose(-2, -1)
print(k_t.shape)
k_t

In [ ]:
sim1 = (q @ k.transpose(-2, -1))/math.sqrt(key_dimension)
sim1
# Attention Scores Matrix

tensor([[[ 0.1055, -0.0482,  0.1911],
         [ 0.3925,  0.4027,  0.4638],
         [ 0.1237,  0.0377,  0.1841]],

        [[ 0.0517, -0.0477,  0.0772],
         [ 0.1894,  0.1613,  0.2007],
         [ 0.1188,  0.0999,  0.1262]]], grad_fn=<DivBackward0>)

In [ ]:
sim2 = F.softmax(sim1, dim=-1)
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

tensor([[[0.2984, 0.2733, 0.3024],
         [0.3977, 0.4290, 0.3972],
         [0.3039, 0.2978, 0.3003]],

        [[0.3108, 0.2948, 0.3143],
         [0.3567, 0.3634, 0.3556],
         [0.3324, 0.3418, 0.3301]]], grad_fn=<SoftmaxBackward0>)

In [ ]:
v # Value vectors

tensor([[[-0.0955,  0.1917],
         [-0.2222,  0.6181],
         [-0.1241,  0.2313]],

        [[-0.3120,  0.2750],
         [-0.1258,  0.5210],
         [-0.4945,  0.3406]]], grad_fn=<ViewBackward0>)

In [ ]:
sim3 = sim2 @ v
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

tensor([[[-0.1267,  0.2961],
         [-0.1825,  0.4333],
         [-0.1324,  0.3118]],

        [[-0.2895,  0.3461],
         [-0.3329,  0.4086],
         [-0.3099,  0.3819]]], grad_fn=<UnsafeViewBackward0>)

In [ ]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our input. Ready to be passed on to the next layer.

torch.Size([2, 3, 4])


tensor([[[ 0.5473,  0.1857,  0.1325, -0.7257],
         [ 0.5019,  0.1961,  0.0726, -0.7626],
         [ 0.5425,  0.1866,  0.1252, -0.7299]],

        [[ 0.4506,  0.2431,  0.2052, -0.7557],
         [ 0.4198,  0.2546,  0.1898, -0.7746],
         [ 0.4355,  0.2480,  0.1935, -0.7660]]], grad_fn=<ViewBackward0>)

In [ ]:
class SelfAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention
    """

    def __init__(self, d_model, key_dimension, num_heads):
        super().__init__()
        self.d_model = d_model
        self.key_dimension = key_dimension
        self.num_heads = num_heads
        self.w_q = nn.Linear(d_model, key_dimension) # * num_heads)
        self.w_k = nn.Linear(d_model, key_dimension) # * num_heads)
        self.w_v = nn.Linear(d_model, key_dimension) # * num_heads)
        self.w_o = nn.Linear(key_dimension, d_model)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        

        sim1 = (q @ k.transpose(-2, -1))/math.sqrt(key_dimension)
        sim2 = F.softmax(sim1, dim=1)
        sim3 = sim2 @ v
        output = w_o(sim3)
        return output

In [4]:
dummy_input = torch.rand(1, 3, 4)
dummy_input

tensor([[[0.4822, 0.5225, 0.2562, 0.3452],
         [0.5920, 0.5045, 0.6581, 0.3271],
         [0.1261, 0.9438, 0.0162, 0.2651]]])